> **仓库整理提示｜2026-11-23**：本篇来自已保存的中文学习版，不保证包含个人电脑上的后续修改。本次未重新执行真实数据；保留输出仅作示例，不是本次复现记录。
>
> 先阅读 [运行指南](../../guides/setup.md) 和 [版本来源](SOURCES.md)。从前往后执行，辅助函数必须先定义再调用。下面开始原中文学习版正文。


# **教程 2：探索与处理 DAS 数据**

> **中文版说明**：本 Notebook 译自 [xdas-dev/tutorials 的 02_exploration&processing.ipynb](https://github.com/xdas-dev/tutorials/blob/main/02_exploration%26processing.ipynb)，原项目采用 [GPL-3.0 许可证](https://github.com/xdas-dev/tutorials/blob/main/LICENSE.md)。代码的运算流程与原版一致；补充了读图说明和面向初学者的提示。

## **中文学习版共创说明**

**稚方Xttus × ChatGPT（Codex）｜共同整理，边学边完善**

这份中文学习版是在实际运行 XDAS 教程、尝试读图和逐步讨论的过程中共同完善的。稚方Xttus负责运行实践、提出学习问题、尝试解释结果并反馈；ChatGPT（Codex）协助汉化、梳理代码与原理、核对资料并整理补充说明。教程中的绝对值求平均实验、同色标滤波对照、重采样目的、FK 读图和速度相关转换解读，都来自这一过程。

我们希望它不仅帮助读者运行代码，也帮助读者说清楚：为什么做这一步、结果表示什么、又有哪些结论暂时不能下。

原教程的代码与基础内容归原项目贡献者所有；本说明记录中文版的合作过程，不替代原作者署名与 GPL-3.0 许可证要求。

*共创署名补充于 2026 年 9 月 30 日。*

## **这份教程要解决什么**

教程 1 已把许多 HDF5 文件整理成可直接打开的汇总文件。本教程从 `outputs/singlecable.nc` 取出一根光纤的记录，完成一次基本的数据分析流程：**选取时空窗口 → 看原始时空图 → 计算通道统计量与去共同模态 → 滤波、重采样 → 查看频率–波数图 → 沿光纤积分**。

| 章节 | 你会做什么 | 得到什么 |
| --- | --- | --- |
| 1. 选取数据 | 用 `sel` 按时间坐标选，用 `isel` 按样本序号选。 | 约 64 秒的事件窗口；写出 `outputs/event.nc`，供教程 3 使用。 |
| 2. 可视化 | 把时间 × 距离的数组画成颜色图。 | 识别随时间变化的纹理和固定位置的异常条纹。 |
| 3. 基础处理 | 求各通道的平均绝对幅值，并估计、减去参考距离段的共同变化。 | 比较处理前后的图像。 |
| 4. 信号处理 | 按物理频率滤波、改变采样率、计算频率–波数图。 | 学会在坐标单位下设置参数。 |
| 5. 沿光纤积分 | 沿距离积分并移除长波长趋势。 | 看变形率／速度的示例结果。 |

> **运行前检查**：在 教程运行根目录（包含 `data/` 和 `outputs/`） 下启动 JupyterLab；教程 1 生成的 `outputs/singlecable.nc` 应已存在。按顺序运行代码单元格（**Shift + Enter**），因为后面的运算会继续使用变量 `da`。图像首次重跑时可能需要一点时间。

In [ ]:
import numpy as np

import xdas as xd

In [ ]:
da = xd.open("outputs/singlecable.nc")
da

## **1. 选取数据**

上一格的 `da` 是教程 1 保存的一根光纤的完整记录。接下来这四行代码依次做：

| 代码 | 这一行做什么 | 结果在哪里 |
| --- | --- | --- |
| `da = da.sel(time=slice("2023-11-03T12:26:40", None))` | `sel` 按**时间坐标**选择：从 12:26:40 开始，一直到这段记录结束；`None` 表示右边界不设限。 | 缩短后的数组重新赋给 `da`。 |
| `da = da.isel(time=slice(None, 4000))` | `isel` 按**样本下标**选择：从刚选出的记录开头取前 4,000 个时间点。 | 再次更新 `da`，它现在是一个较短的时间窗口。 |
| `da.to_netcdf("outputs/event.nc")` | 把当前 `da` 写入 `outputs/event.nc`。 | **教程 3** 会打开这个文件；紧接着的绘图单元格直接使用内存中的变量 `da`。 |
| `da` | 在当前单元格显示最终数组的维度、坐标和数据概况。 | 供你核对选取是否成功。 |

本样本的时间采样率约 62.5 Hz（后文会用代码算出），所以 4,000 个时间点覆盖约 64 秒。这里的 64 秒只是帮助你理解窗口长度，不是额外的筛选条件。

In [ ]:
da = da.sel(time=slice("2023-11-03T12:26:40", None))  # everything after
da = da.isel(time=slice(None, 4000))  # then the first 4000 samples
da.to_netcdf("outputs/event.nc")  # keep it for the next notebook
da

## **2. 可视化：先看原始时空图**

`DataArray` 已带有 `time` 和 `distance` 坐标，因此 `da.plot()` 能自动用时间与距离给图像标轴。下面设 `yincrease=False` 让时间从上到下增加，并用 `vmin`、`vmax` 固定颜色范围；这些选项改变显示方式，不修改 `da` 中的测量值。

In [ ]:
da.plot(yincrease=False, vmin=-1000, vmax=1000)

### **如何读懂刚才的图**

这是一张**单根光纤的时空图**：横轴是沿光纤的距离（约 0–153 km，图上坐标用米），纵轴是时间（约 12:26:40–12:27:44，**越向下越晚**）。图中的每个像素代表一个距离通道在一个时间采样点的测量值。

| 图中现象 | 能直接读出的信息 | 还不能直接下的结论 |
| --- | --- | --- |
| 颜色条从 −1000 到 +1000；绿色附近接近 0，黄色偏正，紫色偏负。 | 颜色表示测量值的符号和显示范围；超过上下限的值会呈现端点颜色。 | 不能把黄色直接解释为“更强的地震”；负值的绝对值也可能很大。 |
| 中部约 20–100 km，12:26:55 后出现弯曲或倾斜的纹理。 | 一些变化在不同距离通道上出现的时间不同，值得后续追踪。 | 单凭这张原始图无法确定事件来源或传播速度。 |
| 约 70 km、105–115 km 有近似竖直的窄条；最左侧与约 120 km 以外的纹理更杂。 | 某些固定距离段的表现与邻近通道不同，分析时应注意通道质量。 | 这些区域的具体成因仍需查看原始通道、光缆位置和处理前后图。 |

**读图练习**：先固定一个距离，沿竖直方向看它随时间怎样变化；再固定一个时刻，沿水平方向看哪些通道同时响应。比较本图和后面滤波后的图时，记得两张图使用的色标范围不同。

## **3. 基础处理：看每个通道有多活跃**

先把 `da` 想成一张表：**每一行是一个时刻，每一列是一个距离通道**。下面这格的目标是回答：在刚才选出的约 64 秒里，*哪一段光纤的测量值通常比较大*？

| 代码部分 | 对这张表做了什么 |
| --- | --- |
| `np.abs(da)` | 每个数先取绝对值，避免正负值直接相加时互相抵消。 |
| `.mean("time")` | 对**每一列**沿时间方向求平均；例如某通道在三个时刻的值为 −2、4、−6，取绝对值后平均为 `(2+4+6)/3=4`。 |

这样会把**时间轴压掉**，得到 `A(distance)`：每个距离通道一个平均绝对幅值。它适合比较不同通道在这个窗口内的整体活跃程度；它不会告诉我们某个事件具体在哪一秒发生。较大值也可能来自噪声或异常通道，仍需回到时空图检查。

In [ ]:
np.abs(da).mean("time")

### **换一个方向求平均：估计共同变化**

这一次想回答的是：*在每个时刻，30–70 km 这一段通道的平均测量值是多少*？因此先用 `.sel(distance=slice(30_000, 70_000))` 选参考段，再用 `.mean("distance")` 对**每一行**沿距离方向求平均。时间轴保留，得到 `common(time)`，即每个时刻一个参考值。这里直接平均带符号的原始值，没有先取绝对值。

举例：某一时刻参考段三个通道为 `2, 4, 6`，此时 `common=4`；从所有通道减去 4 后，参考段变成 `−2, 0, 2`，其平均值为 0。XDAS 根据时间坐标把对应时刻的 `common` 从各个距离通道减去，再画处理后的时空图。参考段的选择会影响结果，它也可能包含想保留的真实信号。

In [ ]:
common = da.sel(distance=slice(30_000, 70_000)).mean("distance")
(da - common).plot(yincrease=False, vmin=-1000, vmax=1000)

### **如何读懂减去 `common` 后的图**

横轴仍是沿光纤距离，纵轴仍是时间；色标也和前面的原始时空图一样是 −1000 到 +1000，因此可以直接比较同一位置的颜色变化。图中的斜向纹理仍可见，而约 70 km、105–115 km 的细竖线以及右端较杂的区域也仍存在。说明减去参考段的**空间平均值**并不会自动消除所有固定通道异常或远端噪声。

这一步精确做的是：对每个时刻 `t`，计算 30–70 km 参考段的平均 `common(t)`，然后从**该时刻所有距离通道**减去这个数。在参考段内，处理后的各通道平均值应接近 0；单个通道仍可能为正或负。图像变化看起来可能不大，因为显示范围固定在 ±1000，且保留下来的局部结构仍很强。

> **解释边界**：这张图只能说明相减后的显示结果。若某种真实信号恰好在参考段的很多通道中同时出现，它的部分成分也会进入 `common(t)` 并被减去；不能仅凭图像相似或不同就断言处理一定更好。

### **动手练习：取绝对值后再求空间平均，会怎样改变图像？**

上面教程的 `common = da.sel(...).mean("distance")` **保留正负号**，用于观察参考区间的共同变化。这里故意先取绝对值，定义另一个变量 `common_abs`：它表示**每个时刻** 30–70 km 参考区间的平均信号幅度。`np.abs` 作用于测量值，不作用于距离坐标；`mean("distance")` 沿距离通道求平均，结果只剩时间轴。先运行下一格看一维曲线，再思考：什么时候 `common_abs` 最大？


In [ ]:
common_abs = np.abs(da.sel(distance=slice(30_000, 70_000))).mean("distance")
common_abs.plot()


**读图**：这段数据的 `common_abs` 在约 12:26:55 开始上升，接近 12:27:00 达到峰值，随后逐渐下降；约 12:27:08 仍高于开始时的水平。它说明参考区间的**平均绝对幅度**在这段时间明显增强，但单凭曲线无法确定源的位置或衰减机制。

你还可以**选择性**运行下面的实验，比较取绝对值前后作为参考值的区别：

```python
(da - common_abs).plot(yincrease=False, vmin=-1000, vmax=1000)
```

如果在峰值附近看到横贯多个距离位置的紫色带，先联系计算方式解释：`common_abs` 始终非负，峰值约 1400；在每个时刻，这个值被从**所有距离通道**的原始有符号信号中减去。即使某通道原始值接近 0，减去 1400 后也会成为负值。这里的 `vmin=-1000` 还会使低于 −1000 的数在色标上显示为同一种最深紫色。因此，横向紫带不能直接证明整根光纤同时出现真实的负向变化。想判断是否有传播，应回看原始时空图中信号随距离移动的斜向纹理。

**小结**：`mean("distance")` 决定了按哪些通道求平均；是否先用 `np.abs`，决定算的是带正负号的平均值还是平均绝对幅度。这两个参考量回答的问题不同，别把后者当作前面教程的共同信号消除方法。


## **4. 信号处理**

XDAS 提供信号处理函数，可以直接使用带物理单位的参数，例如频率（Hz）、时间窗口（秒）或距离窗口（米）。采样率改变后，这些参数仍然表达同一个物理范围。

下面的 `xd.filter(da, (1.0, 10.0), dim="time")` 沿时间轴做 1–10 Hz 带通滤波。代码只把结果放在 `filtered` 中，原变量 `da` 仍供后面的示例使用。滤波后画图时颜色范围改为 ±500，因此比较图像强弱时还要看色标。

In [ ]:
filtered = xd.filter(da, (1.0, 10.0), dim="time")  # bandpass 1-10 Hz
filtered.plot(yincrease=False, vmin=-500, vmax=500)

### **读图练习：滤波前后怎样比较？**

上面的滤波图使用 ±500 色标，便于观察较弱的纹理；前面的原始图使用 ±1000。**比较两图颜色对应的数值时，先统一色标**。你可以在新单元格依次运行：

```python
da.plot(yincrease=False, vmin=-1000, vmax=1000)
filtered.plot(yincrease=False, vmin=-1000, vmax=1000)
```

注意上限是 `vmax=1000`，若误写成 `10000`，零值在色条上的位置就会偏离中部，两图颜色失去可比性。两图坐标、色标一致时，可以逐处观察：

| 观察区域 | 这段记录里可见的变化 | 合理解释与边界 |
| --- | --- | --- |
| 近端约 0–10 km | 原图很强的黄紫竖带在滤波图中明显减弱，剩余细纹理。 | 该处原始信号的一部分被 1–10 Hz 带通抑制；仅凭图片不能断言其具体来源。 |
| 约 20–100 km 的事件纹理 | 约 12:26:55 后的斜向、交替纹理仍可辨认。 | 其中有保留下来的 1–10 Hz 变化；不能只凭斜纹确定事件源或传播速度。 |
| 约 70 km、105–115 km 的竖线 | 滤波后仍可见。 | 固定通道位置的异常或强信号不一定被时间带通滤掉。 |
| 约 120 km 以远 | 仍有较杂的纹理。 | 一部分杂乱波动也处于保留的频带。 |

**读图结论**：这个滤波器对每个距离通道的**时间变化**保留 1–10 Hz 成分，削弱其他频率成分；它不会自动区分有用事件与噪声。色标相同后可以做定性对照，若要严格比较信号大小，还需对选定的时间和距离区间计算数值指标。


### **为什么要重采样？**

DAS 的时间采样密、距离通道多。研究几十千米范围的整体变化时，可能不需要每个原始样本。这里通过降低时间和距离两个方向的采样密度，**减小数据量，让后续运算和绘图更快**；代价是损失部分细节。目标密度应根据研究问题选择。

| 方向 | 原始采样 | 本例目标 | 需要接受的代价 |
| --- | --- | --- | --- |
| 距离 | 相邻通道约 15.32 m | 相邻位置约 245.12 m | 小尺度空间结构更难分辨。 |
| 时间 | 每秒 62.5 个样本 | 每秒 20 个样本 | 较高频率的时间变化不能完整保留。 |

距离样本数约变为原来的 `1/16`，时间样本数约变为原来的 `20/62.5`；两者相乘为 `0.02`，即总样本数约为原来的 **1/50**（边界样本会有少量差异）。例如，看一次事件在长距离上的变化可以尝试这种缩减；研究局部几十米范围的异常则可能需要更密的原始通道。

降低采样密度时，需要考虑**抗混叠**：不能把“每隔几个点直接取一个”与重采样等同，否则过快的变化可能被错误表示成较慢的变化。时间采样降为 20 Hz 后，奈奎斯特频率是 10 Hz，实际分析频带应留在这个上限以下；距离采样变稀也会限制可分辨的空间波长。

重采样时，可以用目标采样率 `rate`，也可以用相邻样本之间的间隔 `interval`。时间轴的采样率单位是 Hz，距离轴的间隔按坐标单位（这里是米）。先用 `xd.get_sampling_interval` 查看原始记录：时间约 62.5 Hz，通道沿光纤约每 15.32 m 一个。


In [ ]:
rate = 1 / xd.get_sampling_interval(da, "time")
spacing = xd.get_sampling_interval(da, "distance")
print(f"{rate:.1f} Hz, one channel every {spacing:.2f} m")

**下面两行按顺序做什么？**

1. 第一行从原始 `da` 出发，沿距离轴重采样，结果放入 `resampled`。
2. 第二行继续处理这个 `resampled`，把时间采样率改为 20 Hz。

注意第一行使用的是 **`da`**，前面的带通滤波结果保存在 **`filtered`** 中。因此本段是在演示另一种处理操作，并没有自动接着带通结果处理。重采样图的色标改为 ±250；与原图比较颜色对应的数值时，仍需统一色标。


In [ ]:
resampled = xd.resample(da, interval=245.12, snap=True, dim="distance")  # 16 -> 1
resampled = xd.resample(resampled, 20, dim="time")  # 62.5 Hz -> 20 Hz

resampled.plot(yincrease=False, vmin=-250, vmax=250)

> **`snap=True` 的含义**：距离轴的目标间隔 245.12 m 恰好是 15.32 m 的 16 倍，因此此处约每 16 个原通道对应一个新位置。`snap=True` 将目标间隔对齐到原始样本格点。时间轴的下一步则把采样率从 62.5 Hz 改为 20 Hz；XDAS 会按相应比例进行重采样。

### **频率–波数（FK）图：为什么要换一种观察方式？**

时空图回答“信号在什么时候、哪个位置出现”；FK 图回答“这段记录由哪些时间频率和空间变化尺度组成”。它把一段时间、距离范围内的信号整体分解成不同的波动成分，因此不能再从图上直接读出某个事件的发生时刻或位置。

只做时间 FFT 能看振动频率；再做距离 FFT，可以同时看信号沿光纤变化的疏密。这样有助于识别占优势的频率、空间波长，以及具有一致传播关系的谱脊线，为后续设计频率–波数滤波提供依据。**下面只计算和显示谱，没有实施 FK 滤波。**

| 对应代码 | 为什么这样做 |
| --- | --- |
| `da.sel(distance=slice(30_000, 70_000))` | 只分析 30–70 km 参考段，避免将整根光纤不同区域全部混在一起；结论仅针对所选区间。 |
| `xd.taper(fk, dim="distance")` | 将距离窗口两端逐渐压低，减弱突然截断导致的频谱泄漏；边缘数据也因此被降权。 |
| `xd.taper(fk, dim="time")` | 对时间窗口两端做同样处理。FFT 的周期延拓会把两端拼接，平滑边界能减弱拼接跳变。 |
| `xd.fft.rfft(... time → frequency)` | 分解时间振荡。原始值为实数，只保留非负时间频率即可表达对应的信息。 |
| `xd.fft.fft(... distance → wavenumber)` | 分解沿距离的空间变化，保留正、负波数，便于研究方向相关结构。 |
| `np.abs(fk)` | 复数谱取模，查看幅值；这一步不再显示相位。 |
| `20 * np.log10(...)` | 压缩幅值的显示范围，使较弱成分也容易看见。得到的是对数谱幅值，不是标定的声压级或归一化功率谱密度。 |
| `fk.plot(...)` | 横轴显示波数，纵轴显示频率；`xlim` 和 `vmin/vmax` 只限定显示范围。`interpolation="antialiased"` 是绘图插值选项，不是在数据上做抗混叠滤波。 |

**变量关系**：本段重新从原始 `da` 出发，没有使用 `filtered` 或 `resampled`。本教程的这些示例各自展示一种处理，不是自动串联的处理流水线。


In [ ]:
fk = da.sel(distance=slice(30_000, 70_000))
fk = xd.taper(fk, dim="distance")
fk = xd.taper(fk, dim="time")
fk = xd.fft.rfft(fk, dim={"time": "frequency"})  # rename "time" -> "frequency"
fk = xd.fft.fft(fk, dim={"distance": "wavenumber"})  # and "distance" -> "wavenumber"
fk = 20 * np.log10(np.abs(fk))

fk.plot(interpolation="antialiased", vmin=90, vmax=130, xlim=(-0.03, 0.03))

### **如何读懂刚生成的 FK 图？**

| 坐标／颜色 | 含义 | 简单例子 |
| --- | --- | --- |
| `frequency`（Hz） | 每秒振荡多少次。 | 5 Hz 对应周期 0.2 秒。 |
| `wavenumber`（周期/米，本例 FFT 坐标约定） | 沿空间变化有多密；看绝对值时，越大表示空间波长越短。 | $\lvert k\rvert=0.01$ 对应 $\lambda=1/\lvert k\rvert=100\,\mathrm{m}$。 |
| 黄／紫 | 对数谱幅值较大／较小。 | 这里紫色不代表原始信号为负，黄色也不代表正向运动。 |

你这次运行的图中，较亮部分主要在约 **2–10 Hz**，部分延伸到更高频率；正、负波数两侧都有幅值较大的区域。它说明这些频率和空间尺度组合比较突出，但不能仅凭左右两片亮区认定有两个振动源，也不能把整片亮区当成一种单独的波。

靠近 $k=0$ 表示较长的空间波长，即沿距离变化较慢；$k=0$ 本身对应空间上不变化的成分。正、负波数与方向相关，但必须结合 FFT 的符号约定与光纤距离坐标的方向解释，不能直接当作地图上的东、西方向。

**进一步理解传播（可选）**：对理想平面行波，若采用周期/米的波数，$|v_{\mathrm{app}}|=f/|k|$。例如 $f=5\,\mathrm{Hz}$、$|k|=0.005\,\mathrm{m}^{-1}$ 对应沿光纤的视速度约 $1000\,\mathrm{m/s}$。此公式应作用于能辨认的行波谱脊线；不能随便从亮区挑一点就认定它是实际波速。弯曲光纤、斜入射或多种波叠加都会影响解释。

**为什么纵轴到约 31 Hz，而前面重采样到了 20 Hz？** 因为这里用的仍是采样率约 62.5 Hz 的原始 `da`，频率上限约为 31.25 Hz。若改用 `resampled`，时间频率上限才会变为 10 Hz，距离方向可表达的波数范围也会随采样间隔变小。

参考：[XDAS FFT 文档](https://xdas.readthedocs.io/en/latest/_autosummary/xdas.fft.fft.html)。


## **5. 从应变率相关测量量到速度：为什么沿距离积分？**

这一步不是让图“更好看”，而是尝试把**光纤的局部伸缩变化**转换成更容易与运动联系起来的量。先考虑已正确标定、与介质耦合良好、近似直线的一段光纤；实际 DAS 还受到标距平均、仪器响应和光纤走向等因素影响。

设 $u(x,t)$ 是沿光纤方向的位移，$v(x,t)=\partial u/\partial t$ 是该方向速度。在上述一维近似下：

$$
\varepsilon(x,t)=\frac{\partial u}{\partial x},\qquad
\dot\varepsilon(x,t)=\frac{\partial v}{\partial x}.
$$

**直观理解**：相邻位置若一起以相同速度运动，光纤没有相对伸缩，DAS 应变率可以接近零；若相邻位置速度不同，二者间的长度会变化，才产生应变率。因此，DAS 的应变率与速度的“空间变化”有关，不能直接当成速度。

沿距离把这些局部变化积累起来：

$$
\int_{x_0}^{x}\dot\varepsilon(\xi,t)\,d\xi
=v(x,t)-v(x_0,t).
$$

这解释了第一行 `xd.integrate(da, dim="distance")`：对**每个时刻**沿距离积分。若输入是真实应变率（$\mathrm{s}^{-1}$），距离单位是米，则积分结果具有 $\mathrm{m/s}$ 的量纲；若输入只是应变率相关的仪器计数，结果也仍需要标定才能换成真实速度单位。

**第一行得到的是参考位置之间的速度差。** 应变率无法确定整段光纤共同运动的速度，积分也无法凭空补回这个信息。代码中的变量名 `deformation` 沿用原教程，但变量名并不决定物理量；这里没有沿时间积分，所以不能把它理解为已经恢复了位移。

### **为什么积分后，还要移除 1000 m 的滑动均值？**

积分会积累误差：假设某时刻应变率含有很小的空间恒定偏差 $b(t)$，积分后就会产生随距离增长的趋势 $b(t)(x-x_0)$。在空间频谱里，积分也容易放大低波数成分，造成很长空间波长的趋势。

第二行 `xd.sliding_mean_removal(deformation, wlen=1000.0, dim="distance")` 在每个时刻估计一个约 **1000 m** 窗口内的局部背景，再从积分结果减去它：

$$
v_{\mathrm{example}}(x,t)=I(x,t)-\overline I_{1000\,\mathrm m}(x,t),
\quad I=\text{沿距离积分的结果}.
$$

可以把它理解为**沿距离方向抑制缓慢变化的背景**。XDAS 的滑动均值默认使用加权窗口，并非简单把每个点同等平均；窗口长度也不是一个精确的波长截止线。

这样做能让局部运动纹理更容易观察，但也会削弱真实的长波长运动。得到的 `velocity` 是经过空间背景去除的速度相关示例，不能宣称恢复了完整的绝对速度。1000 m 是本例的处理尺度，换任务时要根据要保留的空间尺度选择。

| 你想做的转换 | 沿哪个轴积分 | 理想一维关系 |
| --- | --- | --- |
| 应变率 → 应变变化 | 时间 | $\int\dot\varepsilon\,dt$ |
| 应变率 → 速度差 | 距离 | $\int\dot\varepsilon\,dx$ |
| 应变 → 位移差 | 距离 | $\int\varepsilon\,dx$ |

所以，“看到应变率就先沿时间积分”并不总适合当前目标：本例想看速度相关量，直接沿距离积分有相应物理依据。若目标是位移，需要相应的时间积分，并处理积分常数和漂移。

本段也重新从 **`da`** 出发，没有把前面的 FK 图、滤波结果或重采样结果作为输入。

### **参考页打不开时：直接看这里的内容摘要**

XDAS 的参考页标题是 **Convert to displacement（转换为位移）**，它讲的是另一条更长的处理链：

1. 应变率沿**时间**积分，得到应变变化。
2. 应变沿**距离**积分，得到位移差相关的结果。
3. 移除空间滑动均值，抑制积分后的长空间波长背景；该参考页示例窗口为 2000 m。

本教程的目标是速度相关量，所以直接把应变率沿距离积分，再用 1000 m 窗口去局部背景。两者都需要考虑参考值、标定和光纤几何。参考页并不是本教程速度示例的逐行副本；理解上述区别即可继续学习。

外部资料入口：

- [XDAS 转换为位移示例（当前官方地址）](https://xdas.readthedocs.io/en/latest/user-guide/how-to/convert-displacement.html)。官方文档已将本页移入 `user-guide/how-to/` 目录，旧地址返回 404；若外部网页不可访问，可直接阅读上面的摘要和本节原理。
- [XDAS 官方 GitHub 仓库](https://github.com/xdas-dev/xdas)：可从仓库的 `docs`、`xdas` 目录查找文档和实现；版本更新后文件位置与 API 可能变化。
- [滑动均值的实现说明](https://xdas.readthedocs.io/en/latest/_modules/xdas/signal.html)。



In [ ]:
deformation = xd.integrate(da, dim="distance")
velocity = xd.sliding_mean_removal(deformation, wlen=1000.0, dim="distance")

velocity.plot(yincrease=False, vmin=-50_000, vmax=50_000)

### **运行后怎么读这张速度相关图？**

横轴仍是距离、纵轴仍是时间；颜色现在对应沿距离积分并去局部背景后的值，已不是原始应变率相关测量值。可观察斜向纹理是否更容易辨认、是否残留固定距离的异常；积分会传播异常通道的影响，去背景也不能保证消除所有异常。边界附近的结果需要结合窗口处理谨慎解释。

色标改为 ±50,000，量纲也随距离积分改变，因此不能把它与原始图的颜色直接当成同一物理量比较。判断为真实速度之前，需要确认输入单位、标定、光纤几何和耦合情况。

**完成检查**：你能否说明 `mean("time")` 与 `mean("distance")` 各保留哪个轴、FK 图为什么不再表示事件位置、应变率为什么沿距离积分可以得到速度差，以及移除滑动均值会损失哪类真实信号？
